In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
import numpy as np
import cv2
import os
from random import random

device = "cuda" if torch.cuda.is_available() else "cpu" 
# GPU 사용가능시 cuda라는 GPU 사용, 사용불가시 cpu사용
print(device)

# ---------------------------
# 데이터 전처리 함수
# ---------------------------
def getImg(folder, w, h, y):
    trainData, testData, trainLabel, testLabel = [], [], [], []
    for i, f in enumerate(os.listdir(folder)):
        data = cv2.imread(folder + "./" + f, cv2.IMREAD_COLOR)  # color 이미지로 읽기
        data = cv2.resize(data, (w, h)) # 파일의 크기를 지정한 크기로 리사이즈
        data = data / 255.0  # 픽셀 값을 0~1 범위로 정규화 (학습안정화)
        # 3:7 로 test, train 용으로 나눔
        if random() < 0.3:
            testData.append(np.array(data, dtype=np.float32))
            testLabel.append(y[i])
        else:
            trainData.append(np.array(data, dtype=np.float32))
            trainLabel.append(y[i])
    return trainData, trainLabel, testData, testLabel


# ---------------------------
# Dataset 클래스
# ---------------------------
class BunsikDataset(Dataset): 
    def __init__(self, data, label):
        super().__init__()
        self.datas = torch.from_numpy(np.array(data))  
        # (N,H,W,C) (데이터 개수, 세로 픽셀수, 가로 픽셀 수, 채널 수)  c = 3 -> 컬러, 1 -> 흑백
        self.datas = self.datas.permute(0, 3, 1, 2)     
        # 차원 재배열 (N,C,H,W) - GPU 연산에 최적화 된 구조
        self.labels = torch.from_numpy(np.array(label)) # 정수 인덱스만 저장 (원핫X)
    
    def __len__(self):
        return len(self.datas) # 샘플 수 반환

    def __getitem__(self, index):
        return self.datas[index], self.labels[index] # index 번째 샘플 반환


# ---------------------------
# CNN 모델 정의
# ---------------------------
class BunsikMenuCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.bmcnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),  # 3 채널 입력 -> 32채널 출력, 3x3 필터, 패딩1 ->출력 크기 동일
            nn.ReLU(),
            nn.MaxPool2d(2, 2),              # 크기 절반
            
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),              # 크기 절반
            
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)               # 크기 절반
        )
        self.f = nn.Flatten() # 특정 맵(feature map) -> 1차원 벡터로 펴기 (평탄화)
        self.nn = nn.Sequential(
            nn.Linear(128 * (50//8) * (100//8), 256), # 50,100 → MaxPool 3번 → 6,12
            nn.ReLU(),
            nn.Dropout(0.5), # 50프로 날림 (과정합 방지)
            nn.Linear(256, 5)
        )
        
    def forward(self, x):
        x = self.bmcnn(x)
        x = self.f(x)
        x = self.nn(x)
        return x


# ---------------------------
# 데이터 준비
# ---------------------------
label = ["떡볶이", "오뎅", "김밥", "튀김", "순대"]
yData = [0,0,0,0,0, 1,1,1,1,1, 2,2,2,2,2, 3,3,3,3,3, 4,4,4,4,4]

trainData, trainLabel, testData, testLabel = getImg("./torch_ex/bunsikMenu", 100, 50, yData)
# 이미지 경로와 리사이즈 할 크기 정답
trainDataset = BunsikDataset(trainData, trainLabel)
testDataset = BunsikDataset(testData, testLabel)
trainDataLoader = DataLoader(trainDataset, 8, shuffle=True)  # batchSize=8
testDataLoader = DataLoader(testDataset, 8, shuffle=False)


# ---------------------------
# 학습 준비
# ---------------------------
model = BunsikMenuCNN().to(device)
lossFn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ---------------------------
# 학습 루프
# ---------------------------
epoch = 20
for t in range(epoch):
    model.train() # 학습 모드로 전환
    total_loss = 0
    for x, y in trainDataLoader:
        x, y = x.to(device), y.to(device)
        
        pred = model(x)
        loss = lossFn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    print(f"Epoch {t+1}, Loss: {total_loss / len(trainDataLoader):.4f}")

# ---------------------------
# 정확도 평가
# ---------------------------
model.eval()
size = len(testDataLoader.dataset)
ok = 0
with torch.no_grad(): # 평가모드(기울기 계산x)
    for x, y in testDataLoader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        ok += (pred.argmax(1) == y).type(torch.float32).sum().item()
print(f"정확도: {ok/size:.2f}")


cpu
Epoch 1, Loss: 1.6512
Epoch 2, Loss: 1.3183
Epoch 3, Loss: 1.3845
Epoch 4, Loss: 1.7021
Epoch 5, Loss: 1.6961
Epoch 6, Loss: 1.5446
Epoch 7, Loss: 1.5155
Epoch 8, Loss: 1.3743
Epoch 9, Loss: 1.7826
Epoch 10, Loss: 1.3839
Epoch 11, Loss: 1.2418
Epoch 12, Loss: 1.2659
Epoch 13, Loss: 1.8326
Epoch 14, Loss: 1.1263
Epoch 15, Loss: 1.4181
Epoch 16, Loss: 0.9882
Epoch 17, Loss: 0.5952
Epoch 18, Loss: 0.3687
Epoch 19, Loss: 0.4409
Epoch 20, Loss: 0.4724
정확도: 0.88


In [ ]:
# 코드 전체를 한 줄씩 설명

# import torch
# from torch.utils.data import Dataset, DataLoader
# from torch import nn, optim
# import numpy as np
# import cv2
# import os
# from random import random


# 필요한 라이브러리 임포트

# torch : PyTorch 핵심

# Dataset, DataLoader : 데이터셋과 배치 처리

# nn : 신경망 모듈

# optim : 최적화 알고리즘

# numpy : 배열 연산

# cv2 : OpenCV, 이미지 처리

# os : 폴더/파일 접근

# random : 학습/테스트 데이터 랜덤 분할

# device = "cuda" if torch.cuda.is_available() else "cpu" 
# print(device)


# GPU가 사용 가능하면 "cuda"를, 아니면 "cpu"를 사용

# 학습 속도 향상을 위해 GPU 사용 여부 확인

# def getImg(folder, w, h, y):
#     trainData, testData, trainLabel, testLabel = [], [], [], []


# 이미지 전처리 함수 정의

# 학습용/테스트용 데이터와 라벨을 담을 리스트 초기화

#     for i, f in enumerate(os.listdir(folder)):
#         data = cv2.imread(folder + "./" + f, cv2.IMREAD_COLOR)


# 지정 폴더 내 모든 파일(f)을 순회

# cv2.imread로 컬러 이미지 읽기

#         data = cv2.resize(data, (w, h))
#         data = data / 255.0


# 이미지를 (w,h) 크기로 리사이즈

# 0~255 픽셀 값을 0~1 범위로 정규화

#         if random() < 0.3:
#             testData.append(np.array(data, dtype=np.float32))
#             testLabel.append(y[i])
#         else:
#             trainData.append(np.array(data, dtype=np.float32))
#             trainLabel.append(y[i])


# 랜덤으로 학습/테스트 데이터 분리 (약 30% → 테스트, 70% → 학습)

# np.float32로 변환 후 리스트에 추가

#     return trainData, trainLabel, testData, testLabel


# 학습/테스트 데이터와 라벨 반환

# class BunsikDataset(Dataset): 
#     def __init__(self, data, label):
#         super().__init__()
#         self.datas = torch.from_numpy(np.array(data))  
#         self.datas = self.datas.permute(0, 3, 1, 2)     
#         self.labels = torch.from_numpy(np.array(label))


# PyTorch Dataset 상속

# 데이터 리스트를 torch.Tensor로 변환

# permute : (N,H,W,C) → (N,C,H,W)

# 라벨도 Tensor로 변환

#     def __len__(self):
#         return len(self.datas)


# 데이터셋 샘플 개수 반환

#     def __getitem__(self, index):
#         return self.datas[index], self.labels[index]


# 인덱스에 해당하는 데이터와 라벨 반환 (DataLoader가 배치 처리 시 사용)

# class BunsikMenuCNN(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.bmcnn = nn.Sequential(
#             nn.Conv2d(3, 32, 3, padding=1),
#             nn.ReLU(),
#             nn.MaxPool2d(2, 2),
            
#             nn.Conv2d(32, 64, 3, padding=1),
#             nn.ReLU(),
#             nn.MaxPool2d(2, 2),
            
#             nn.Conv2d(64, 128, 3, padding=1),
#             nn.ReLU(),
#             nn.MaxPool2d(2, 2)
#         )


# CNN 모델 정의

# 3개의 Conv+ReLU+MaxPool 블록

# 입력 채널 3 (RGB) → 32 → 64 → 128 채널

# MaxPool로 공간 크기 절반

#         self.f = nn.Flatten()


# CNN feature map을 1차원 벡터로 변환

#         self.nn = nn.Sequential(
#             nn.Linear(128 * (50//8) * (100//8), 256),
#             nn.ReLU(),
#             nn.Dropout(0.5),
#             nn.Linear(256, 5)
#         )


# Fully Connected 레이어 정의

# 입력 128 * 6 * 12 → 256 → 5 (클래스 수)

# Dropout으로 과적합 방지

#     def forward(self, x):
#         x = self.bmcnn(x)
#         x = self.f(x)
#         x = self.nn(x)
#         return x


# 순전파 정의

# CNN → Flatten → FC → 출력

# label = ["떡볶이", "오뎅", "김밥", "튀김", "순대"]
# yData = [0,0,0,0,0, 1,1,1,1,1, 2,2,2,2,2, 3,3,3,3,3, 4,4,4,4,4]
# trainData, trainLabel, testData, testLabel = getImg("./torch_ex/bunsikMenu", 100, 50, yData)


# 라벨 정의

# 이미지 불러와서 학습/테스트 데이터 생성

# trainDataset = BunsikDataset(trainData, trainLabel)
# testDataset = BunsikDataset(testData, testLabel)
# trainDataLoader = DataLoader(trainDataset, 8, shuffle=True)
# testDataLoader = DataLoader(testDataset, 8, shuffle=False)


# Dataset 객체 생성

# DataLoader로 배치 처리 (배치 크기 8)

# 학습용 데이터는 섞기(shuffle=True), 테스트용은 그대로

# model = BunsikMenuCNN().to(device)
# lossFn = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)


# 모델 GPU/CPU 이동

# 손실 함수 CrossEntropyLoss (분류용)

# Adam optimizer

# epoch = 20
# for t in range(epoch):
#     model.train()
#     total_loss = 0
#     for x, y in trainDataLoader:
#         x, y = x.to(device), y.to(device)
#         pred = model(x)
#         loss = lossFn(pred, y)
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()
#     print(f"Epoch {t+1}, Loss: {total_loss / len(trainDataLoader):.4f}")


# 학습 루프

# 모델을 학습 모드로 전환

# 배치마다 순전파 → 손실 계산 → 역전파 → 가중치 업데이트

# 배치 손실 합산 후 평균 출력

# model.eval()
# size = len(testDataLoader.dataset)
# ok = 0
# with torch.no_grad():
#     for x, y in testDataLoader:
#         x, y = x.to(device), y.to(device)
#         pred = model(x)
#         ok += (pred.argmax(1) == y).type(torch.float32).sum().item()
# print(f"정확도: {ok/size:.2f}")


# 평가 모드

# 기울기 계산 OFF (torch.no_grad) → 메모리 절약

# 배치마다 예측값과 정답 비교, 맞춘 수 합산

# 전체 데이터 정확도 출력

In [ ]:
# 예측할 이미지 파일
img_path = "test.png"

# 1. 이미지 읽기 및 전처리
img = cv2.imread(img_path, cv2.IMREAD_COLOR)
img = cv2.resize(img, (100, 50))        # 학습 시 사용한 (w,h)와 동일하게
img = img.astype(np.float32)            # float32 변환
img = np.transpose(img, (2, 0, 1))      # (H,W,C) -> (C,H,W)
img = np.expand_dims(img, 0)            # 배치 차원 추가: (1,3,50,100)
img_tensor = torch.from_numpy(img).to(device)

# 2. 모델 예측
model.eval()
with torch.no_grad():
    logits = model(img_tensor)          # 로짓 출력
    probs = torch.softmax(logits, dim=1)  # 확률 계산
    pred_class = probs.argmax(1).item()   # 예측 클래스 인덱스

# 3. 클래스 라벨 출력
label = ["떡볶이", "오뎅", "김밥", "튀김", "순대"]
print(f"예측 클래스 인덱스: {pred_class}")
print(f"예측 메뉴: {label[pred_class]}")

예측 클래스 인덱스: 1
예측 메뉴: 오뎅
